In [146]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy import interpolate

data = [
# replace this with the input
]
if not data: raise ValueError("No Data")
rank = data[0].replace("_", " ")
rank = rank.replace("legend", "1k") if "top" in rank else rank.replace("diamond ", "d").replace("to", "-").replace(" legend", "l").replace("1", "d1")
df = pd.DataFrame(data[1])

class_colors = {
    "Death Knight": "#6C699A",
    "DK": "#6C699A",
    "Demon Hunter": "#256F3D",
    "DH": "#256F3D",
    "Druid": "#FF7F0E",
    "Hunter": "#2CA02C",
    "Mage": "#17BECF",
    "Paladin": "#F0BD27",
    "Priest": "#C7C7C7",
    "Rogue": "#7F7F7F",
    "Shaman": "#2B7DB4",
    "lock": "#A27099",
    "Warlock": "#A27099",
    "Warrior": "#C81518"
}
df["class"] = df["name"].str.extract(f"({"|".join(class_colors.keys())})")
df["color"] = df["class"].map(class_colors).fillna("black")
df["class"] = df["class"].replace("lock", "Warlock").replace("Death Knight", "DK")
df["is_hl"] = df["name"].str.contains("HL|Highlander")

In [ ]:
df1 = df.copy()

df1 = df1[df1["winrate"] >= 40]
df1 = df1[df1["pop"] >= 200]
#df1 = df1[df1["pop"] <= 300]
#df1 = df1[df1["is_hl"]]
#df1 = df1[df1["class"] == "Druid"]
#df1 = df1[df1["winrate"] == df1.groupby("class")["winrate"].transform("max")]

plt.figure(figsize=(10, 6))
plt.scatter(df1["pop"], df1["winrate"], c=df1["color"], s=90, edgecolor="black", alpha=0.9, zorder=3)
plt.xlabel(f"# of games ({rank} this patch)")
plt.xscale("log")
plt.ylabel("Win Rate (%)")
l = [m*z for e in range(len(str(min(df1["pop"])))-1,len(str(max(df1["pop"])))) for m in range(1,10) if min(df1["pop"])-(z:=10**e) < m*z < max(df1["pop"])+z]
plt.xticks(l, l)
r = range(round(min(df1["winrate"])) - 1, round(max(df1["winrate"])) + 2)
plt.yticks(r)
if 50 in r:
    plt.axhline(y=50, color="black", linewidth=0.6)
plt.grid()

for _, row in df1.iterrows():
    plt.annotate(row["name"], (row["pop"], row["winrate"]), fontsize=10, alpha=0.7)

plt.show()

In [ ]:
class_pops = df.groupby("class", as_index=False)["pop"].sum()

class_pops["class"] = class_pops["class"].where(class_pops["pop"] / class_pops["pop"].sum() >= 0.05, "Other")
class_pops = class_pops.groupby("class", as_index=False)["pop"].sum().sort_values("pop", ascending=False)

plt.pie(class_pops["pop"], labels=class_pops["class"], autopct=lambda p: f"{p:.1f}".rstrip(".0") + "%",
        colors=class_pops["class"].map(class_colors).fillna("pink"), wedgeprops={"edgecolor": "black", "linewidth": 0.35})
plt.title(f"Share of classes ({rank} this patch)")
plt.show()